Imports and global configuration

In [21]:
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelBinarizer

from sklearn.tree import DecisionTreeClassifier as SkDecisionTree
from sklearn.ensemble import RandomForestClassifier as SkRandomForest
from sklearn.ensemble import ExtraTreesClassifier as SkExtraTrees
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


Utility functions

In [22]:
def entropy(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

def gini(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return 1 - np.sum(ps ** 2)

def information_gain(y, y_left, y_right):
    H_parent = entropy(y)
    w_left = len(y_left) / len(y)
    w_right = len(y_right) / len(y)
    return H_parent - (w_left * entropy(y_left) + w_right * entropy(y_right))


Custom Decision Tree Implementation

In [23]:
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

class CustomDecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, n_features=None, random_state=42):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.random_state = random_state
        self.root = None

    def fit(self, X, y):
        self.n_classes = len(np.unique(y))
        self.n_total_features = X.shape[1]
        if self.n_features is None:
            self.n_features = self.n_total_features
        self.root = self._grow_tree(X, y, depth=0)

    def _best_split(self, X, y):
        rng = np.random.RandomState(self.random_state)
        feat_idxs = rng.choice(self.n_total_features, self.n_features, replace=False)

        best_gain = -1
        split_idx, split_thresh = None, None

        for feat in feat_idxs:
            thresholds = np.unique(X[:, feat])
            for thresh in thresholds:
                left_idx = X[:, feat] <= thresh
                right_idx = X[:, feat] > thresh
                if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                    continue
                gain = information_gain(y, y[left_idx], y[right_idx])
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat
                    split_thresh = thresh

        return split_idx, split_thresh

    def _grow_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        num_labels = len(np.unique(y))

        if (depth >= self.max_depth or
            num_labels == 1 or
            num_samples < self.min_samples_split):
            leaf_value = Counter(y).most_common(1)[0][0]
            return TreeNode(value=leaf_value)

        feat, thresh = self._best_split(X, y)
        if feat is None:
            leaf_value = Counter(y).most_common(1)[0][0]
            return TreeNode(value=leaf_value)

        left_idx = X[:, feat] <= thresh
        right_idx = X[:, feat] > thresh

        left = self._grow_tree(X[left_idx], y[left_idx], depth + 1)
        right = self._grow_tree(X[right_idx], y[right_idx], depth + 1)

        return TreeNode(feature=feat, threshold=thresh, left=left, right=right)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


Custom Random Forest Implementation

In [24]:
class CustomRandomForest:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None, random_state=42):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        rng = np.random.RandomState(self.random_state)

        for i in range(self.n_trees):
            idxs = rng.choice(len(X), len(X), replace=True)
            X_sample = X[idxs]
            y_sample = y[idxs]

            tree = CustomDecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=self.n_features,
                random_state=self.random_state + i
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.array([Counter(tree_preds[:, i]).most_common(1)[0][0]
                         for i in range(X.shape[0])])


Custom Extra Trees Implementation

In [25]:
class CustomExtraTrees:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, n_features=None, random_state=42):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        rng = np.random.RandomState(self.random_state)

        for i in range(self.n_trees):
            idxs = rng.choice(len(X), len(X), replace=True)
            X_sample = X[idxs]
            y_sample = y[idxs]

            tree = CustomExtraTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=self.n_features,
                random_state=self.random_state + i
            )
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        return np.array([Counter(tree_preds[:, i]).most_common(1)[0][0]
                         for i in range(X.shape[0])])


class CustomExtraTree(CustomDecisionTree):
    def _best_split(self, X, y):
        rng = np.random.RandomState(self.random_state)
        feat_idxs = rng.choice(self.n_total_features, self.n_features, replace=False)

        split_idx, split_thresh = None, None

        for feat in feat_idxs:
            min_val, max_val = X[:, feat].min(), X[:, feat].max()
            thresh = rng.uniform(min_val, max_val)

            left_idx = X[:, feat] <= thresh
            right_idx = X[:, feat] > thresh
            if len(y[left_idx]) == 0 or len(y[right_idx]) == 0:
                continue

            split_idx = feat
            split_thresh = thresh
            break

        return split_idx, split_thresh


In [26]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")

    lb = LabelBinarizer()
    y_test_bin = lb.fit_transform(y_test)
    y_pred_bin = lb.transform(y_pred)

    try:
        auc = roc_auc_score(y_test_bin, y_pred_bin, average="macro")
    except:
        auc = np.nan

    return acc, f1, auc


In [27]:
def run_experiment(dataset_loader, dataset_name):
    X, y = dataset_loader(return_X_y=True)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_SEED
    )

    results = []

    model = CustomDecisionTree(
        max_depth=3,
        min_samples_split=2,
        n_features=X.shape[1],
        random_state=RANDOM_SEED
    )
    acc, f1, auc = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(["Custom Decision Tree", acc, f1, auc])

    model = CustomRandomForest(
        n_trees=10,
        max_depth=3,
        min_samples_split=2,
        n_features=int(np.sqrt(X.shape[1])),
        random_state=RANDOM_SEED
    )
    acc, f1, auc = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(["Custom Random Forest", acc, f1, auc])

    model = CustomExtraTrees(
        n_trees=10,
        max_depth=3,
        min_samples_split=2,
        n_features=int(np.sqrt(X.shape[1])),
        random_state=RANDOM_SEED
    )
    acc, f1, auc = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(["Custom Extra Trees", acc, f1, auc])

    model = SkDecisionTree(random_state=RANDOM_SEED, max_depth=5)
    acc, f1, auc = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(["sklearn Decision Tree", acc, f1, auc])

    model = SkRandomForest(n_estimators=10, max_depth=5, random_state=RANDOM_SEED)
    acc, f1, auc = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(["sklearn Random Forest", acc, f1, auc])

    model = SkExtraTrees(n_estimators=10, max_depth=5, random_state=RANDOM_SEED)
    acc, f1, auc = evaluate_model(model, X_train, X_test, y_train, y_test)
    results.append(["sklearn Extra Trees", acc, f1, auc])

    df = pd.DataFrame(results, columns=["Model", "Accuracy", "F1-score", "AUROC"])
    print(f"\nResults for {dataset_name}")
    display(df)

    return X_train, X_test, y_train, y_test, df


X_train_iris, X_test_iris, y_train_iris, y_test_iris, df_iris = run_experiment(load_iris, "Iris Dataset")
X_train_wine, X_test_wine, y_train_wine, y_test_wine, df_wine = run_experiment(load_wine, "Wine Dataset")


Results for Iris Dataset


,Model,Accuracy,F1-score,AUROC
0,Custom Decision Tree,0.966667,0.965899,0.97271
1,Custom Random Forest,0.966667,0.965899,0.97271
2,Custom Extra Trees,0.966667,0.965899,0.97271
3,sklearn Decision Tree,1.000000,1.000000,1.00000
4,sklearn Random Forest,1.000000,1.000000,1.00000
5,sklearn Extra Trees,1.000000,1.000000,1.00000



Results for Wine Dataset


,Model,Accuracy,F1-score,AUROC
0,Custom Decision Tree,0.916667,0.889360,0.914773
1,Custom Random Forest,0.944444,0.952137,0.961039
2,Custom Extra Trees,0.722222,0.706739,0.771104
3,sklearn Decision Tree,0.944444,0.942474,0.952110
4,sklearn Random Forest,0.972222,0.966284,0.971591
5,sklearn Extra Trees,0.972222,0.968046,0.982143


In [28]:
X_train, X_test, y_train, y_test, df_wine = run_experiment(load_iris, "Iris Dataset")


Results for Iris Dataset


,Model,Accuracy,F1-score,AUROC
0,Custom Decision Tree,0.966667,0.965899,0.97271
1,Custom Random Forest,0.966667,0.965899,0.97271
2,Custom Extra Trees,0.966667,0.965899,0.97271
3,sklearn Decision Tree,1.000000,1.000000,1.00000
4,sklearn Random Forest,1.000000,1.000000,1.00000
5,sklearn Extra Trees,1.000000,1.000000,1.00000


Plot setup

In [29]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.tree import plot_tree

os.makedirs("plots", exist_ok=True)
plt.style.use("seaborn-v0_8-whitegrid")


In [30]:

feature_names = [f"Feature {i}" for i in range(X_train.shape[1])]

custom_dt = CustomDecisionTree(
    max_depth=3,
    min_samples_split=2,
    n_features=X_train.shape[1],
    random_state=RANDOM_SEED
)
custom_dt.fit(X_train, y_train)

custom_rf = CustomRandomForest(
    n_trees=50,
    max_depth=5,
    min_samples_split=2,
    n_features=int(np.sqrt(X_train.shape[1])),
    random_state=RANDOM_SEED
)
custom_rf.fit(X_train, y_train)

custom_et = CustomExtraTrees(
    n_trees=50,
    max_depth=5,
    min_samples_split=2,
    n_features=int(np.sqrt(X_train.shape[1])),
    random_state=RANDOM_SEED
)
custom_et.fit(X_train, y_train)


In [31]:

def compute_forest_importance(forest, n_features):
    importances = np.zeros(n_features)
    for tree in forest.trees:
        stack = [tree.root]
        while stack:
            node = stack.pop()
            if node is None or node.value is not None:
                continue
            importances[node.feature] += 1
            stack.append(node.left)
            stack.append(node.right)
    return importances / importances.sum()

dt_importance = compute_forest_importance(CustomRandomForest(n_trees=1, random_state=RANDOM_SEED), X_train.shape[1])
rf_importance = compute_forest_importance(custom_rf, X_train.shape[1])
et_importance = compute_forest_importance(custom_et, X_train.shape[1])

top_idx = np.argsort(rf_importance)[-10:]
top_features = [feature_names[i] for i in top_idx]

x = np.arange(len(top_idx))
width = 0.25

plt.figure(figsize=(12, 6))
plt.bar(x - width, dt_importance[top_idx], width, label="Decision Tree")
plt.bar(x, rf_importance[top_idx], width, label="Random Forest")
plt.bar(x + width, et_importance[top_idx], width, label="Extra Trees")

plt.xticks(x, top_features, rotation=45, ha="right")
plt.xlabel("Features")
plt.ylabel("Normalized Importance")
plt.title("Top 10 Feature Importance Comparison Across Models")
plt.legend()

plt.tight_layout()
plt.savefig("plots/feature_importance_comparison.png", dpi=300, bbox_inches="tight")  # FIX ADDED HERE
plt.close()


/tmp/ipython-input-3136980008.py:15: RuntimeWarning: invalid value encountered in divide
  return importances / importances.sum()


In [32]:
models = {
    "Decision Tree": custom_dt,
    "Random Forest": custom_rf,
    "Extra Trees": custom_et
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - {name}")

    plt.tight_layout()
    filename = f"plots/confusion_matrix_{name.replace(' ', '_').lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()


In [33]:
performance = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    performance.append([name, acc, f1])

df_perf = pd.DataFrame(performance, columns=["Model", "Accuracy", "F1-score"])

x = np.arange(len(df_perf))
width = 0.35

plt.figure(figsize=(8, 6))
plt.bar(x - width/2, df_perf["Accuracy"], width, label="Accuracy")
plt.bar(x + width/2, df_perf["F1-score"], width, label="F1-score")

plt.xticks(x, df_perf["Model"], rotation=20)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.legend()

plt.tight_layout()
plt.savefig("plots/model_performance_comparison.png", dpi=300, bbox_inches="tight")
plt.close()


In [34]:
sk_dt_vis = SkDecisionTree(
    max_depth=3,
    random_state=RANDOM_SEED
)
sk_dt_vis.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(
    sk_dt_vis,
    feature_names=feature_names,
    class_names=[str(c) for c in np.unique(y_train)],
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Decision Tree Structure (max_depth = 3)")
plt.tight_layout()
plt.savefig("plots/decision_tree_structure.png", dpi=300, bbox_inches="tight")
plt.close()


In [35]:
tree_counts = [1, 5, 10, 20, 50]
rf_scores = []
et_scores = []

for n in tree_counts:
    rf = CustomRandomForest(n_trees=n, max_depth=5, n_features=int(np.sqrt(X_train.shape[1])), random_state=RANDOM_SEED)
    et = CustomExtraTrees(n_trees=n, max_depth=5, n_features=int(np.sqrt(X_train.shape[1])), random_state=RANDOM_SEED)

    rf.fit(X_train, y_train)
    et.fit(X_train, y_train)

    rf_scores.append(accuracy_score(y_test, rf.predict(X_test)))
    et_scores.append(accuracy_score(y_test, et.predict(X_test)))

plt.figure(figsize=(8, 6))
plt.plot(tree_counts, rf_scores, marker="o", label="Random Forest")
plt.plot(tree_counts, et_scores, marker="o", label="Extra Trees")

plt.xlabel("Number of Trees")
plt.ylabel("Accuracy")
plt.title("Effect of Ensemble Size on Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig("plots/ensemble_size_vs_accuracy.png", dpi=300, bbox_inches="tight")
plt.close()


In [36]:
depths = [1, 2, 3, 5, 10]
train_acc = []
test_acc = []

for d in depths:
    dt = CustomDecisionTree(max_depth=d, n_features=X_train.shape[1], random_state=RANDOM_SEED)
    dt.fit(X_train, y_train)

    train_acc.append(accuracy_score(y_train, dt.predict(X_train)))
    test_acc.append(accuracy_score(y_test, dt.predict(X_test)))

plt.figure(figsize=(8, 6))
plt.plot(depths, train_acc, marker="o", label="Train Accuracy")
plt.plot(depths, test_acc, marker="o", label="Test Accuracy")

plt.xlabel("Tree Depth")
plt.ylabel("Accuracy")
plt.title("Bias–Variance Tradeoff in Decision Trees")
plt.legend()

plt.tight_layout()
plt.savefig("plots/bias_variance_tradeoff.png", dpi=300, bbox_inches="tight")
plt.close()


In [37]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_train_2d = pca.fit_transform(X_train)
X_test_2d = pca.transform(X_test)

print("Original feature dimension:", X_train.shape[1])
print("Reduced to 2D for decision boundary visualization.")


Original feature dimension: 4
Reduced to 2D for decision boundary visualization.


In [38]:
from tqdm import tqdm

def plot_decision_boundary(
    model,
    X,
    y,
    title,
    filename,
    resolution=0.05,
    batch_size=10000
):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, resolution),
        np.arange(y_min, y_max, resolution)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = np.zeros(len(grid), dtype=int)

    n_batches = int(np.ceil(len(grid) / batch_size))

    for i in tqdm(range(n_batches), desc=f"Predicting grid for {title}"):
        start = i * batch_size
        end = min((i + 1) * batch_size, len(grid))
        Z[start:end] = model.predict(grid[start:end])

    Z = Z.reshape(xx.shape)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.Set1)
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", cmap=plt.cm.Set1)

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.title(title)

    handles, _ = scatter.legend_elements()
    plt.legend(handles, [str(c) for c in np.unique(y)], title="Classes")

    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.close()


In [39]:
dt_2d = CustomDecisionTree(
    max_depth=5,
    min_samples_split=2,
    n_features=2,
    random_state=RANDOM_SEED
)
dt_2d.fit(X_train_2d, y_train)

rf_2d = CustomRandomForest(
    n_trees=10,
    max_depth=5,
    min_samples_split=2,
    n_features=2,
    random_state=RANDOM_SEED
)
rf_2d.fit(X_train_2d, y_train)

et_2d = CustomExtraTrees(
    n_trees=10,
    max_depth=5,
    min_samples_split=2,
    n_features=2,
    random_state=RANDOM_SEED
)
et_2d.fit(X_train_2d, y_train)

plot_decision_boundary(
    dt_2d,
    X_train_2d,
    y_train,
    title="Decision Boundary - Custom Decision Tree (PCA 2D)",
    filename="plots/decision_boundary_decision_tree.png"
)

plot_decision_boundary(
    rf_2d,
    X_train_2d,
    y_train,
    title="Decision Boundary - Custom Random Forest (PCA 2D)",
    filename="plots/decision_boundary_random_forest.png"
)

plot_decision_boundary(
    et_2d,
    X_train_2d,
    y_train,
    title="Decision Boundary - Custom Extra Trees (PCA 2D)",
    filename="plots/decision_boundary_extra_trees.png"
)


Predicting grid for Decision Boundary - Custom Decision Tree (PCA 2D): 100%|██████████| 2/2 [00:00<00:00, 149.09it/s]
Predicting grid for Decision Boundary - Custom Random Forest (PCA 2D): 100%|██████████| 2/2 [00:00<00:00,  9.75it/s]
Predicting grid for Decision Boundary - Custom Extra Trees (PCA 2D): 100%|██████████| 2/2 [00:00<00:00,  8.57it/s]


In [40]:
import shutil
from google.colab import files

shutil.make_archive("plots", "zip", "plots")
files.download("plots.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>